# OpenRouter model screening before Tinker training

One secret-carrying answer per APPS problem/model, at most **100 shared problems**.
Compare functional, exact-message, and joint pass@1. Inference is on OpenRouter;
later training would be on Tinker. This notebook does not train models.

**Preparation is free of inference calls.** It saves all requests and estimates cost.
The later execution cell asks yes/no before spending on OpenRouter and Modal.
See [README.md](README.md) for model availability, limitations, and artifact schemas.

In [1]:
import json
import os
import random
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Locate the repository when Jupyter starts in this notebook's subdirectory.
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "ciphers/variable_naming_in_python_v2").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ciphers.variable_naming_in_python_v2.data.apps import REPO_ROOT
from ciphers.variable_naming_in_python_v2.data.codex_apps import SecretTask
from ciphers.variable_naming_in_python_v2.decoder import CipherConfig
from ciphers.variable_naming_in_python_v2.tinker.prepare import RunConfig, artifact_path, prepare_run
from ciphers.variable_naming_in_python_v2.tinker.evaluate import run_prepared, summarize

if not all(os.environ.get(key) for key in ("OPENROUTER_API_KEY", "STEGO_ARTIFACTS_DIR")):
    env_path = REPO_ROOT / ".env"
    if not env_path.is_file():
        raise FileNotFoundError("Set the environment variables or provide the repository-root .env")
    load_dotenv(env_path, override=False)
for key in ("OPENROUTER_API_KEY", "STEGO_ARTIFACTS_DIR"):
    if not os.environ.get(key):
        raise RuntimeError(f"Missing required environment variable: {key}")

/opt/miniconda3/envs/stego/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cipher_path = REPO_ROOT / "ciphers/variable_naming_in_python_v2/tinker/official_cipher.json"
secret = SecretTask(
    cipher=CipherConfig.model_validate_json(cipher_path.read_text()),
    message_bits="101",
)
config = RunConfig(secret=secret, num_problems=100, max_tokens=16384, estimated_output_tokens=4096)
config

RunConfig(secret=SecretTask(cipher=CipherConfig(special_variables={'sequence': ('values', 'numbers'), 'size': ('count', 'size'), 'accumulator': ('total', 'sum_value'), 'minimum': ('lowest', 'minimum'), 'maximum': ('highest', 'maximum'), 'result': ('answer', 'result')}, control_bits=1, length_bits=2), message_bits='101'), models=('nvidia/nemotron-3.5-lightning', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'nvidia/nemotron-3-nano-30b-a3b', 'qwen/qwen3.8-27b', 'qwen/qwen3.6-35b-a3b', 'qwen/qwen3.5-9b', 'qwen/qwen3.5-4b', 'thinkingmachines/inkling-small'), num_problems=100, seed=42, apps=AppsConfig(split='train', difficulties=('introductory',), min_lines=20, max_lines=None, min_chars=0, max_chars=None, min_tests=10, revision='0f10e424e13e1c2a69f851e153097b71b6734a1f', cache_dir=PosixPath('datasets/apps')), max_tokens=16384, estimated_output_tokens=4096, timeout_s=300)

## Prepare and review

The next cell downloads the filtered APPS dataset and public model prices, writes all
requests, then costs the saved prompts. It sends **no inference requests**.
To review an existing run, replace the prepare call with `Path("variable_naming_v2/tinker/<run-id>")`.

The estimate assumes 4,096 output tokens per answer; the limit scenario assumes
16,384, including reasoning where the provider includes it in the limit.
Neither is a hard spending ceiling. Modal grading and account fees are excluded.

In [3]:
run_dir = prepare_run(config)
print("Run directory:", artifact_path(run_dir))
estimate = json.loads((artifact_path(run_dir) / "estimate.json").read_text())
price_rows = []
for row in estimate["models"]:
    pricing = row["pricing"]
    price_rows.append({
        "model": row["model"], "available": row["available"], "requests": row["requests"],
        "input_characters": row["input_characters"], "estimated_input_tokens": row["input_tokens"],
        "input_USD_per_M": pricing["prompt"] * 1e6 if pricing else None,
        "output_USD_per_M": pricing["completion"] * 1e6 if pricing else None,
        "estimated_USD": row["estimated_usd"], "limit_scenario_USD": row["limit_scenario_usd"],
    })
display(pd.DataFrame(price_rows))
print(f"Estimated inference cost: ${estimate['estimated_usd']:.2f}")
print(f"Output-limit scenario: ${estimate['limit_scenario_usd']:.2f}")
print("Inspect requests.jsonl, grading_cases.json, and config.json in the run directory.")

Skipped 1 malformed APPS rows


Run directory: /Users/4gate/git/StegoICMLMechInterp2026/artifacts/variable_naming_v2/tinker/20260915T232923Z-382b25f4


,model,available,requests,input_characters,estimated_input_tokens,input_USD_per_M,output_USD_per_M,estimated_USD,limit_scenario_USD
0,nvidia/nemotron-3.5-lightning,True,100,1555570,520154,0.080,0.20,0.123532,0.369292
1,openai/gpt-oss-120b,True,100,1555570,520154,0.037,0.17,0.088878,0.297774
2,openai/gpt-oss-20b,True,100,1555570,520154,0.030,0.13,0.068853,0.228597
3,nvidia/nemotron-3-nano-30b-a3b,True,100,1555570,520154,0.050,0.20,0.107928,0.353688
4,qwen/qwen3.8-27b,True,100,1555570,520154,0.214,2.55,1.155793,4.289233
5,qwen/qwen3.6-35b-a3b,True,100,1555570,520154,0.100,0.90,0.420655,1.526575
6,qwen/qwen3.5-9b,True,100,1555570,520154,0.100,0.15,0.113455,0.297775
7,qwen/qwen3.5-4b,False,0,0,0,NaN,NaN,0.000000,0.000000
8,thinkingmachines/inkling-small,True,100,1555570,520154,0.450,1.20,0.725589,2.200149


Estimated inference cost: $2.80
Output-limit scenario: $9.56
Inspect requests.jsonl, grading_cases.json, and config.json in the run directory.


## Preview saved prompts

Randomly sample three request rows from the saved `requests.jsonl` file and display
their actual prompts, with model and problem IDs. The seed makes the sample repeatable;
different model rows can contain the same problem. This cell makes no API calls.

In [ ]:
request_rows = [
    json.loads(line)
    for line in (artifact_path(run_dir) / "requests.jsonl").read_text().splitlines()
]
sampled_requests = random.Random(config.seed).sample(request_rows, k=min(3, len(request_rows)))
for request_row in sampled_requests:
    display(Markdown(f"### APPS {request_row['problem_id']} — {request_row['body']['model']}"))
    for message in request_row["body"]["messages"]:
        display(Markdown(message["content"]))

## Optional paid execution

Run only after reviewing the saved requests and estimate. Answer `yes` to start;
anything else sends no inference requests. Execution is sequential, uses one answer
per problem/model, and saves results incrementally. An infrastructure error stops
the run. Started runs cannot be rerun automatically.

In [ ]:
answer = input(f"Run {sum(row['requests'] for row in estimate['models'])} saved requests? "
               f"Estimate ${estimate['estimated_usd']:.2f}; output-limit scenario "
               f"${estimate['limit_scenario_usd']:.2f}, plus Modal. Type yes/no: ").strip().lower()
if answer == "yes":
    results = run_prepared(run_dir, approved=True)
else:
    print("No inference requests sent.")

In [ ]:
# Safe to run before inference or after an interrupted run; incomplete rates stay blank.
display(pd.DataFrame(summarize(run_dir)))